# WTI Crude Oil Price Forecasting — Stateless Methods: Systematic Backtest (Notebook 4 of 7)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) across the geopolitical price shock and its
   aftermath — measuring adaptive real-time responsiveness and calibration.
   The eval window runs through the most recent origin whose 21-business-day
   horizon still resolves against cached data (see `scripts/fetch_wti.py`).

The line-up spans three families behind one `Predictor` interface: **baselines**
(Naive, AutoARIMA), **numerical ML** (LightGBM ± a leak-safe covariate panel),
and **LLM/agent** methods (LLM-process forecasters and a news-reading analyst
agent) — the last run on *both* project models, `gemini-3.1-flash-lite-preview`
and `gemini-3.5-flash`. Every predictor is one toggle line in the registry in
Section 2. Agent configs come from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [17]:
import warnings
from pathlib import Path

from datetime import datetime
import energy_oil_forecasting
import pandas as pd
import yaml
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from energy_oil_forecasting.data import (
    DEFAULT_WTI_COVARIATE_SERIES_IDS,
    WTI_SERIES_ID,
    build_wti_multivariate_service,
)

from energy_oil_forecasting.analyst_agent.agent_original import (
    build_wti_news_config as build_wti_news_config_original,
    build_wti_agent_predictor as build_wti_agent_predictor_original,
)


warnings.filterwarnings("ignore")

# ── Mode ──────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True to run a 2-origin, 1-sample version of the notebook
# for fast local development and end-to-end CI testing. The full specs run
# 51 backtest + 8 eval origins; smoke runs 2 + 2.
SMOKE_TEST = False 

# ── Models ────────────────────────────────────────────────────────────────────
# The project standardises on two Vector-proxy models. Every LLM and agent
# predictor below is run once per model so we can compare them head-to-head.
# (bare proxy names — no "gemini/" prefix)
MODELS = [LITE_MODEL, ADVANCED_MODEL]  # "gemini-3.1-flash-lite-preview", "gemini-3.5-flash"

# ── Derived settings (do not edit below) ─────────────────────────────────────
N_SAMPLES = 1 if SMOKE_TEST else 3  # trajectories per LLMP-Sampled call

# LightGBM hyperparameters (shared by the univariate and +covariate variants).
LAGS = 21  # one trading month of lagged target/covariate history
NUM_SAMPLES_LGBM = 100 if SMOKE_TEST else 200  # Monte-Carlo draws for quantiles
LGBM_KWARGS = {"num_threads": 1, "n_jobs": 1, "verbosity": -1}  # deterministic, quiet

# Data service: WTI target + a leak-safe covariate panel (all Yahoo Finance —
# Brent, natural gas, gasoline, gold, USD index, the USL/USO futures-curve
# contango proxy, and VIX). Non-covariate predictors simply ignore the extras,
# so one service feeds the whole leaderboard. Unavailable tickers are skipped
# with a warning, so this still runs offline / under partial connectivity.
data_service = build_wti_multivariate_service()
COVARIATES = [c for c in DEFAULT_WTI_COVARIATE_SERIES_IDS if c in set(data_service.series_ids)]

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
if SMOKE_TEST:
    backtest_file, eval_file = "energy_oil_smoke.yaml", "energy_oil_eval_smoke.yaml"
else:
    backtest_file, eval_file = "energy_oil_backtest.yaml", "energy_oil_eval.yaml"

with open(spec_dir / backtest_file) as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
with open(spec_dir / eval_file) as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))



# ── Extend to a 10yr backtest / 2yr protected holdout, quarterly cadence ────
# In SMOKE_TEST mode, use a small recent window instead so the notebook can
# be smoke-tested quickly. spec_id gets its own namespace so this never
# collides with (or silently reuses) the biweekly/original cached results.
_suffix = "_smoke" if SMOKE_TEST else ""

full_df = data_service.get_series(WTI_SERIES_ID, as_of=datetime.now()).sort_values("timestamp")
data_start = full_df["timestamp"].min()

# ── ANCHOR_END: the origin grid must not drift with the calendar ─────────────
# Everything below is derived from this one date, so taking it from
# full_df["timestamp"].max() made the whole grid a function of *when the cell
# ran*. Two runs a fortnight apart produced origins offset by ~10 business
# days — and because the prediction cache is keyed by spec_id alone (see
# aieng/forecasting/evaluation/artifacts.py: the window is neither part of the
# key nor checked on load), those mismatched grids landed in the same folder
# and were later compared as if they were the same experiment. They were not.
#
# 2026-07-23 is the value that reproduces the grid the cached agent runs in
# data/predictions/energy_oil_backtest_10yr_quarterly/ were actually computed
# on: backtest 2014-04-14 → 2024-07-23, eval 2024-07-23 → 2026-04-27.
# Change it only if you intend to invalidate that cache, and if you do, delete
# the cached files too — stale files are silently reloaded, not recomputed.
#
# (The separate 10yr cache from the local machine sits under the
# ..._localrun/ spec_id, which was run against ANCHOR_END = 2026-08-06. It is
# namespaced apart precisely so it can never be mistaken for this grid.)
ANCHOR_END = pd.Timestamp("2026-07-23")
data_end = full_df["timestamp"].max() if SMOKE_TEST else ANCHOR_END

holdout_start = data_end - pd.DateOffset(years=2)
holdout_end = data_end
backtest_start = data_start + (holdout_start - data_start) / 2
backtest_end = holdout_start

if SMOKE_TEST:
    backtest_spec.start = (data_end - pd.DateOffset(years=5)).strftime("%Y-%m-%d")
    backtest_spec.end = (data_end - pd.DateOffset(years=5) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    backtest_spec.stride = 5

    eval_spec.start = (data_end - pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    eval_spec.end = data_end.strftime("%Y-%m-%d")
    eval_spec.stride = 5
else:
    backtest_spec.start = backtest_start.strftime("%Y-%m-%d")
    backtest_spec.end = backtest_end.strftime("%Y-%m-%d")
    backtest_spec.stride = 63

    eval_spec.start = holdout_start.strftime("%Y-%m-%d")
    eval_spec.end = (holdout_end - pd.tseries.offsets.BDay(63)).strftime("%Y-%m-%d")
    eval_spec.stride = 63

backtest_spec.tasks[0].horizons = [5, 10, 21, 63]
backtest_spec.spec_id = f"energy_oil_backtest_10yr_quarterly{_suffix}"

eval_spec.tasks[0].horizons = [5, 10, 21, 63]
eval_spec.spec_id = f"energy_oil_eval_10yr_quarterly{_suffix}"

# Guard: the cache cannot detect a grid mismatch, so assert the one we expect.
if not SMOKE_TEST:
    assert (backtest_spec.start, backtest_spec.end) == ("2014-04-14", "2024-07-23"), (
        f"Backtest grid drifted to {backtest_spec.start} → {backtest_spec.end}. "
        "The cached agent results were computed on 2014-04-14 → 2024-07-23; "
        "scoring against a different grid silently compares different experiments."
    )




print(f"{'⚡ SMOKE MODE' if SMOKE_TEST else '📊 FULL MODE'} — MODELS={MODELS}  N_SAMPLES={N_SAMPLES}")
print(f"Covariates registered ({len(COVARIATES)}): {', '.join(COVARIATES) or '(none)'}")
print()
print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))

📊 FULL MODE — MODELS=['gemini-3.1-flash-lite-preview', 'gemini-3.5-flash']  N_SAMPLES=3
Covariates registered (7): brent_log_ret_1b_l1b, natgas_log_ret_1b_l1b, gasoline_log_ret_1b_l1b, gold_log_ret_1b_l1b, dollar_index_log_ret_1b_l1b, oil_curve_contango_l1b, vix_level_l1b

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOADED SPECIFICATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MultiTargetBacktestSpec (spec_id=energy_oil_backtest_10yr_quarterly)
  description: Weekly rolling backtest in 2025 for daily WTI crude oil price forecasting. Evaluates trajectory forecasts (5, 10, 21 business days) with CRPS/MAE and binary up-shock forecasts (climb > $5 in 5 business days) with Brier Score. Used to select the top contender models.
  start:       2014-04-14
  end:         2024-07-23
  stride:      63
  warmup:      250
  tasks:       1

Task: wti_oil_price_forecast
  description: WTI Crude Oil continuous front-month futures Close pric

---
## 2. Candidate Predictors

This experiment puts a full slate of methods on the same `Predictor` interface and
the same rolling backtest, spanning three families:

| Family | Predictors | Role |
|---|---|---|
| **Baselines** | `Naive (Last Value)`, `AutoARIMA` | Carry-forward floor + the classical statistical anchor |
| **Numerical ML** | `LightGBM`, `LightGBM + cov` (+ optional `Prophet`) | Gradient-boosted quantile regression on lagged price (and a leak-safe covariate panel — Brent, gas, gasoline, gold, USD index, the futures-curve contango proxy, and VIX). LightGBM-with-covariates was the strongest method in the S&P 500 study. |
| **LLM / Agent** | `LLMP-Sampled`, `LLMP-Grid`, `News Agent` — each on **both** project models | LLM-process forecasters and a news-reading analyst agent, run on `gemini-3.1-flash-lite-preview` *and* `gemini-3.5-flash` |

The predictor cell below is a **registry**: every method is one line with an
`enabled` flag. Flip a flag to add or drop a predictor — the rest of the
notebook (backtest, scoring, eval, scorecard) iterates over whatever is active.
The two baselines are flagged `baseline=True` and are the only results written to
`adaptive_agent/curriculum/` for Notebooks 5–6, so toggling the others never
disturbs the downstream training data.

In [18]:
from dataclasses import dataclass
from typing import Callable

from aieng.forecasting.methods import (
    LastValuePredictor,
    QuantileGridLLMPredictor,
    QuantileGridLLMPredictorConfig,
    SampledTrajectoryLLMPredictor,
    SampledTrajectoryLLMPredictorConfig,
)
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor
from aieng.forecasting.methods.numerical.darts_regression import DartsLightGBMPredictor
from aieng.forecasting.methods.numerical.error_correction_regression import (
    ErrorCorrectionRegressionPredictor,
)
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_news_config,
    build_wti_news_contrarian_config,
    build_wti_news_factors_v2_config,
    build_wti_news_scenario_schema_config,
    build_wti_scenario_schema_predictor,
)
from energy_oil_forecasting.cfm_agent_v_5_2 import (
    build_cfm_agent_config,
    build_cfm_agent_predictor,
)
from energy_oil_forecasting.analyst_agent.agent_original import (
    build_wti_agent_predictor as build_wti_agent_predictor_original,
    build_wti_news_config as build_wti_news_config_original,
)
from energy_oil_forecasting.cfm_agent_v_5_2_2_delta_governed import (
    build_cfm_agent_config_delta_governed,
    build_cfm_agent_predictor_delta_governed,
)
from energy_oil_forecasting.scenario_schema_anchored import (
    build_wti_news_scenario_schema_anchored_config,
    build_wti_scenario_schema_anchored_predictor,
)
from energy_oil_forecasting.prophet_baseline import ProphetPredictor


@dataclass
class PredictorEntry:
    """One row in the experiment. Flip ``enabled`` to switch a predictor on/off."""

    name: str
    factory: Callable[[], object]  # lazy — built only when enabled
    enabled: bool = True
    baseline: bool = False  # baselines are saved to curriculum/ for NB05–06


# LLM / agent factories — each takes a model so the same recipe runs on both.
# LLMP-Sampled optionally serializes the covariate panel into the prompt
# (labeled exogenous-series blocks); the others are target-only. A distinct
# variant_tag keeps the +cov run separate in the cache and on the leaderboard.
def _llmp_sampled(model, covariates=None):
    return SampledTrajectoryLLMPredictor(
        SampledTrajectoryLLMPredictorConfig(
            model=model,
            n_samples=N_SAMPLES,
            covariate_series_ids=covariates,
            variant_tag="cov" if covariates else None,
        )
    )


def _llmp_grid(model):
    return QuantileGridLLMPredictor(QuantileGridLLMPredictorConfig(model=model))


#def _news_agent(model):
#    return build_wti_agent_predictor(build_wti_news_config(model=model))


#def _news_agent_contrarian(model):
#    return build_wti_agent_predictor(build_wti_news_contrarian_config(model=model))


def _news_agent(model):
    return build_wti_agent_predictor(
        build_wti_news_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                              )
    )


def _news_agent_contrarian(model):
    return build_wti_agent_predictor(
        build_wti_news_contrarian_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )


def _news_agent_factors_v2(model):
    return build_wti_agent_predictor(
        build_wti_news_factors_v2_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )

def _news_agent_scenario_schema(model):
    return build_wti_scenario_schema_predictor(
        build_wti_news_scenario_schema_config(model=model)
    )

def _news_agent_original(model):
    return build_wti_agent_predictor_original(build_wti_news_config_original(model=model))

def _cfm_agent_v_5_2():
    config = build_cfm_agent_config()
    return build_cfm_agent_predictor(config)


def _cfm_agent_v_5_2_2_delta_governed():
    config = build_cfm_agent_config_delta_governed()
    return build_cfm_agent_predictor_delta_governed(config)


def _news_agent_scenario_schema_anchored(model):
    config = build_wti_news_scenario_schema_anchored_config(model=model)
    return build_wti_scenario_schema_anchored_predictor(config)


# ── Experiment registry ───────────────────────────────────────────────────────
# Toggle `enabled` on any line to include/exclude that predictor.
#
# The enabled/disabled flags below are evidence-driven, not habit. Every number
# quoted comes from ONE origin grid (backtest 2014-04-14 → 2024-07-23, 43
# origins / 167 predictions; eval 2024-07-23 → 2026-04-27, 8 origins), scored by
# scripts/run_autoarima_agent_grid.py. Mean CRPS across all horizons, and the
# paired difference against AutoARIMA (negative = beats it):
#
#                              backtest  eval    vs AutoARIMA, backtest / eval
#   News Agent Scenario Schema   3.65    4.21    -0.62 p<0.0001 / -0.70 p=0.0028
#   News Agent Scenario          3.78    5.17    -0.49 p<0.0001 / +0.26 p=0.86
#   News Agent (original)        3.80    4.65    -0.47 p<0.0001 / -0.26 p=0.42
#   AutoARIMA                    4.26    4.91    —
#   ECM (base)                   4.34    4.95    +0.08 p<0.0001 / +0.04 p=0.06
#   Kalman                       4.98    5.73    +0.71 p=0.0003  / +0.82 p=0.20
#   Naive                        5.75    6.62    +1.48 / +1.71
#   LightGBM + cov               loses to Naive at every horizon (own grid)
#
# Read the eval column, not the backtest one. Every agent beats AutoARIMA
# in-sample; only Scenario Schema still does out-of-sample.
from aieng.forecasting.methods.numerical.darts_classical import DartsKalmanForecasterPredictor

REGISTRY = [
    # Floor and reference points. Free, and Naive anchors every comparison.
    PredictorEntry("Naive (Last Value)", LastValuePredictor, enabled=True, baseline=True),
    PredictorEntry("Kalman", DartsKalmanForecasterPredictor, enabled=True, baseline=True),

    # Best token-free method, and the best-calibrated method in the whole
    # line-up: 78–88% coverage against a nominal 80% at every horizon.
    PredictorEntry("AutoARIMA", DartsAutoARIMAPredictor, enabled=True),

    # ON — the same AutoARIMA fitted on log(price) instead of price. Because
    # AutoARIMA picks its own differencing order, fitting on log levels makes it
    # a model of log differences (returns), so its innovation variance scales
    # with the price level rather than being constant in dollars. For a series
    # that has traded from under $20 to over $140 in this window, that is the
    # better-specified assumption.
    #
    # Kam's independent level-vs-return study finds return specifications
    # preferable within every model family, with AutoARIMA Return the strongest
    # statistical point-forecast model overall (best MAE at 1/2/4 weeks, best
    # one-week RMSE, correlation and CRPS), though its edge over the benchmarks
    # narrows materially beyond one week. That study reports calibration
    # explicitly for Kalman but not for ARIMA, and calibration is what this grid
    # is set up to measure: the level version above holds 80.5/80.5/81.0/88.4%
    # coverage at h=5/10/21/63 against a nominal 80%, which is the best in the
    # line-up and is what both anchored agents build on. Whether the return
    # version keeps that is the open question, so both run side by side here
    # rather than one replacing the other.
    #
    # The agents deliberately still anchor on the level version this run.
    # Switching their anchor at the same time as testing whether they preserve
    # ARIMA's calibration would change two things at once.
    PredictorEntry(
        "AutoARIMA (log)",
        lambda: DartsAutoARIMAPredictor(log_transform=True),
        enabled=True,
    ),

    # OFF — redundant with AutoARIMA. Ties it on CRPS (+0.08 backtest, +0.04
    # eval) while winning under a third of paired points, and needs a
    # seven-series covariate panel to do it. Its four "expanded" variants are
    # not reproducible here: they need nine series this data.py lacks, and the
    # expanded/levelonly pair records byte-identical metadata, so whatever
    # separated them was never committed. On the local grid expansion moved
    # CRPS by 0.06, so little is lost.
    PredictorEntry(
        "ECM (base)",
        lambda: ErrorCorrectionRegressionPredictor(covariate_series_ids=COVARIATES),
        enabled=False,
    ),

    # OFF — loses to Naive at every horizon, nothing significant, for
    # ~185 s/origin. Naive is the floor AutoARIMA clears by 1.48 CRPS.
    PredictorEntry(
        "LightGBM + cov",
        lambda: DartsLightGBMPredictor(
            lags=LAGS, lags_past_covariates=LAGS, covariate_series_ids=COVARIATES,
            num_samples=NUM_SAMPLES_LGBM, lgbm_kwargs=LGBM_KWARGS,
        ),
        enabled=False,
    ),

    PredictorEntry(f"News Agent ({LITE_MODEL})", lambda: _news_agent(LITE_MODEL), enabled=False),

    # OFF — spends tokens for no measured out-of-sample gain over a free
    # method (eval -0.26 CRPS, p=0.42). Its in-sample win does not survive.
    PredictorEntry(
        f"News Agent Original ({LITE_MODEL})",
        lambda: _news_agent_original(LITE_MODEL),
        enabled=False,
    ),

    # ON — the only agent that earns its tokens. Beats AutoARIMA by ~14% on
    # both windows and is the sole method holding its edge out of sample (84%
    # of paired eval points). Its weakness is calibration: 48–67% coverage
    # against a nominal 80%, so it wins on median accuracy while being
    # overconfident. Widening its intervals is the open lever.
    PredictorEntry(
        f"News Agent Scenario Schema ({LITE_MODEL})",
        lambda: build_wti_scenario_schema_predictor(build_wti_news_scenario_schema_config(model=LITE_MODEL)),
        enabled=True,
    ),

    # CFM Agent v5.2: policy-controlled forecaster with gemini-3.1-flash-lite
    # optimization (2026.08.19). Compare against Scenario Schema on quarterly grid
    # (same origin dates for fair comparison).
    PredictorEntry(
        "CFM Agent v5.2",
        _cfm_agent_v_5_2,
        enabled=True,
    ),

    # ON — the two agents built to stop the LLM inventing numbers, on the only
    # grid long enough to test that claim. The open question this run exists to
    # answer is calibration, not CRPS: over 2014-2024 AutoARIMA holds 80.5 /
    # 80.5 / 81.0 / 88.4% coverage against a nominal 80% at h=5/10/21/63, while
    # News Agent Scenario Schema -- built on that same AutoARIMA -- gets 56.1 /
    # 53.7 / 69.0 / 62.8%. Its misses are balanced (33 above, 33 below) with
    # near-zero bias, so it is not mis-pointing; it narrows intervals it has no
    # basis to narrow, which is the "widening its intervals is the open lever"
    # note above, measured.
    #
    # Delta-Governed gates the LLM to a discrete rank clipped by evidence tier,
    # and Anchored pins the centre to a deterministic AutoARIMA anchor shifted
    # only as far as real historical price moves justify. Both exist to prevent
    # exactly that narrowing. Whether they actually preserve AutoARIMA's
    # calibration is untested outside a single 25-prediction 2026 window, which
    # has already produced two conclusions that did not survive contact with a
    # second regime.
    PredictorEntry(
        "CFM Agent v5.2.2 Delta-Governed",
        _cfm_agent_v_5_2_2_delta_governed,
        enabled=True,
    ),
    PredictorEntry(
        f"News Agent Scenario Schema Anchored ({LITE_MODEL})",
        lambda: _news_agent_scenario_schema_anchored(LITE_MODEL),
        enabled=True,
    ),
]

# Instantiate only the enabled predictors (lazy factories skip the rest).
PREDICTORS = {e.name: e.factory() for e in REGISTRY if e.enabled}
_BASELINE_PREDICTORS = {e.name for e in REGISTRY if e.baseline}

print(f"Active predictors ({len(PREDICTORS)}):")
for name in PREDICTORS:
    tag = "  (baseline → curriculum/)" if name in _BASELINE_PREDICTORS else ""
    print(f"  {name}{tag}")

Active predictors (5):
  Naive (Last Value)  (baseline → curriculum/)
  Kalman  (baseline → curriculum/)
  AutoARIMA
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview)
  CFM Agent v5.2


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [ ]:
import time

print(f"Running rolling backtest ({backtest_spec.start} → {backtest_spec.end}, {len(PREDICTORS)} predictor(s))...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

backtest_results: dict[str, object] = {}
for i, (_name, _predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)  # pace between predictors, not just after a failure
    backtest_results[_name] = cached_multi_backtest(
        _predictor, backtest_spec, data_service,
        max_retries=4,     # was 2
        retry_delay=30.0,  # was 2.0 — long enough to clear an RPM window
        force_refresh=False,
    )
    print(f"  {_name} ✓")

print(f"\nBacktest complete ({backtest_spec.start} → {backtest_spec.end}).")

Running rolling backtest (2014-04-14 → 2024-07-23, 5 predictor(s))...
LLM/agent runs are expensive — first run will take several minutes.

  Naive (Last Value) ✓
  Kalman ✓
  AutoARIMA ✓
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview) ✓
22:08:52 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 99, in _process_log_task
    await asyncio.wait_for(
  File "/usr/lib/python3.12/asyncio/tasks.py", line 519, in wait_for
    async with timeouts.timeout(timeout):
  File "/usr/lib/python3.12/asyncio/timeouts.py", line 115, in __aexit__
    raise TimeoutError from 

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff28fe20c0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff17049b70>, 24785.604925629)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff2a149610>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff14524050>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff1704acf0>, 24774.4269723)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff14527c80>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff146145f0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff1704bc40>, 25275.282284902)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff146172f0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff14759400>
Unclosed connector
connecti

22:26:00 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 99, in _process_log_task
    await asyncio.wait_for(
  File "/usr/lib/python3.12/asyncio/tasks.py", line 519, in wait_for
    async with timeouts.timeout(timeout):
  File "/usr/lib/python3.12/asyncio/timeouts.py", line 115, in __aexit__
    raise TimeoutError from exc_val
TimeoutError

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

22:32:01 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Trac

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff303033b0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff301999b0>, 31372.554313051)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff303033e0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b20b890>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff30199fd0>, 31366.093390709)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff0b208050>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b5b3b60>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff0b52d9b0>, 31713.411662748)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff0b5b3bc0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b21d400>
Unclosed connector
connec

23:28:26 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 99, in _process_log_task
    await asyncio.wait_for(
  File "/usr/lib/python3.12/asyncio/tasks.py", line 519, in wait_for
    async with timeouts.timeout(timeout):
  File "/usr/lib/python3.12/asyncio/timeouts.py", line 115, in __aexit__
    raise TimeoutError from exc_val
TimeoutError

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

23:34:51 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Trac

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b1e48f0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff0b52fee0>, 32392.226219696)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff0b1e7380>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b262240>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff17991a90>, 32759.030390688)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff0b54b5c0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b2d1ca0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x77ff17990280>, 32752.752185741)])']
connector: <aiohttp.connector.TCPConnector object at 0x77ff0b2d36b0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x77ff0b5bb020>
Unclosed connector
connec


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

00:00:49 - LiteLLM:ERROR: logging_worker.py:104 - LoggingWorker error: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 99, in _process_log_task
    await asyncio.wait_for(
  File "/usr/lib/python3.12/asyncio/tasks.py", line 519, in wait_for
    async with timeouts.timeout(timeout):
  File "/usr/lib/python3.12/asyncio/timeouts.py", line 115, in __aexit__
    raise TimeoutError from exc_val
TimeoutError


---
## 4. Performance Characterisation

We score every active predictor on the 2025 backtest data:
- **CRPS** (Continuous Ranked Probability Score) — sharpness + calibration combined
- **MAE at h=21d** — point forecast accuracy at the longest horizon

The leaderboard ranks the families against each other — how much structure the
numerical methods (AutoARIMA, LightGBM ± covariates) extract over the naive
floor, whether the covariate panel earns its keep, and how the LLM/agent methods
compare across the two models. Where each method wins and where it struggles in
2025 is exactly the material the adaptive agent learns from in Notebook 5.

In [1]:
import math

import numpy as np
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.analysis import score_backtest_results


leaderboard_rows = []

for name, results in backtest_results.items():
    scores = score_backtest_results(results, data_service)

    # score_backtest_results' "mae_h21" is actually MAE blended across every
    # horizon in the task — its mae_horizon=21 parameter isn't wired to any
    # filtering inside it. Computing a real per-horizon breakdown here
    # instead, since horizon 63 (quarterly) is the point of this run and
    # shouldn't be silently averaged in under a "21d" label.
    horizon_errors: dict[int, list[float]] = {h: [] for h in backtest_spec.tasks[0].horizons}
    for result in results.values():
        task = result.spec.task
        offset = pd.tseries.frequencies.to_offset(task.frequency)
        actual_df = data_service.get_series(task.target_series_id, as_of=datetime.now())
        actual_by_date = {
            pd.Timestamp(row["timestamp"]).normalize(): float(row["value"]) for _, row in actual_df.iterrows()
        }
        for pred in result.predictions:
            if not isinstance(pred.payload, ContinuousForecast):
                continue
            fd = pd.Timestamp(pred.forecast_date).normalize()
            actual = actual_by_date.get(fd)
            if actual is None:
                continue
            as_of = pd.Timestamp(pred.as_of)
            for h in task.horizons:
                if (as_of + offset * h).normalize() == fd:
                    horizon_errors[h].append(abs(pred.payload.point_forecast - actual))
                    break

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "80% CI Coverage": scores.get("coverage_80", float("nan")),
    }
    for h in backtest_spec.tasks[0].horizons:
        errs = horizon_errors[h]
        row[f"MAE h={h}d"] = float(np.mean(errs)) if errs else float("nan")
    leaderboard_rows.append(row)

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print(f"BACKTEST PERFORMANCE SUMMARY ({backtest_spec.start} → {backtest_spec.end}):")
print("━" * 72)
print(df_leaderboard.to_string())

kalman_crps = df_leaderboard.loc["Kalman", "Mean CRPS"] if "Kalman" in df_leaderboard.index else float("nan")
naive_crps = (
    df_leaderboard.loc["Naive (Last Value)", "Mean CRPS"]
    if "Naive (Last Value)" in df_leaderboard.index
    else float("nan")
)
if not math.isnan(kalman_crps) and not math.isnan(naive_crps):
    print(
        f"\nKalman CRPS improvement over Naive: {naive_crps - kalman_crps:.4f} "
        f"({(naive_crps - kalman_crps) / naive_crps:.1%})"
    )

NameError: name 'backtest_results' is not defined

In [ ]:
# ── Skipped: shared curriculum/ files feed NB05 ("explores 2025 data") and
# NB06 — this notebook's backtest_spec is the 10yr/quarterly window, not 2025
# weekly, so writing here would silently corrupt their training input.
#
# _CURRICULUM_DIR = Path("adaptive_agent/curriculum")
# _CURRICULUM_DIR.mkdir(exist_ok=True)
# for _name, _result_dict in backtest_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"backtest_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in backtest_results)} backtest result(s) to {_CURRICULUM_DIR}/")

In [ ]:
from collections import Counter
from energy_oil_forecasting.analysis import _business_horizon
import pandas as pd

for name in ["Kalman", "Naive (Last Value)",
             "News Agent Scenario Schema (gemini-3.1-flash-lite-preview)",
             "CFM Agent v5.2"]:  # ← ADDED CFM v5.2 instead
    result = next(iter(backtest_results[name].values()))
    origins = sorted({pd.Timestamp(p.as_of).date() for p in result.predictions})
    horizons_present = Counter(
        _business_horizon(pd.Timestamp(p.as_of), pd.Timestamp(p.forecast_date)) for p in result.predictions
    )
    print(f"{name}: origins={origins}")
    print(f"  horizon counts: {dict(horizons_present)}")

---
## 5. 2026 Evaluation — Held-Out Test Period

We run every active predictor on **18 weekly origins spanning Feb–Jun 2026**
(`energy_oil_eval.yaml`) — the major geopolitical volatility spike not seen
during the 2025 backtest, plus its aftermath. The window runs through the
most recent origin that still fully resolves against cached WTI data (the
21-business-day horizon needs data 21 business days past the origin).

This evaluation serves two purposes:
1. **Measure out-of-sample robustness** — do the 2025 edges (statistical,
   covariate, or LLM/agent) hold under a structural regime shift?
2. **Establish the stateless baseline** that the trained adaptive agents in
   Notebook 6 are compared against. The baseline predictors' results are saved
   to `adaptive_agent/curriculum/` for Notebooks 5 and 6 to load.

In [ ]:
import time

print(f"Running evaluation ({eval_spec.start} → {eval_spec.end}, {len(PREDICTORS)} predictor(s))...")
eval_results: dict[str, object] = {}
for i, (name, predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)
    eval_results[name] = cached_multi_backtest(
        predictor, eval_spec, data_service,
        max_retries=4,
        retry_delay=30.0,
        force_refresh=False,
    )
    print(f"  {name} ✓")

print(f"\nEvaluation complete ({eval_spec.start} → {eval_spec.end}).")

In [ ]:
# ── Skipped: this notebook's eval_spec is the 10yr/quarterly holdout, not
# the original 2026 shock-period window NB05/NB06 expect from these files.
# Writing here would silently overwrite their curriculum inputs.
#
# for _name, _result_dict in eval_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"eval_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in eval_results)} eval result(s) to {_CURRICULUM_DIR}/")

---
## 6. Scorecard

Out-of-sample performance of every active predictor on the 2026 eval period.
These numbers are the **stateless baseline** the adaptive agent variants must
beat in Notebook 6 to demonstrate that training added value.

In [ ]:
import numpy as np
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.analysis import score_backtest_results


scorecard_rows = []
for name in PREDICTORS:
    if name not in eval_results:
        continue
    results = eval_results[name]
    scores = score_backtest_results(results, data_service)

    # Same reasoning as the backtest leaderboard: score_backtest_results'
    # "mae_h21" is blended across every horizon, not horizon-21-specific.
    horizon_errors: dict[int, list[float]] = {h: [] for h in eval_spec.tasks[0].horizons}
    for result in results.values():
        task = result.spec.task
        offset = pd.tseries.frequencies.to_offset(task.frequency)
        actual_df = data_service.get_series(task.target_series_id, as_of=datetime.now())
        actual_by_date = {
            pd.Timestamp(row["timestamp"]).normalize(): float(row["value"]) for _, row in actual_df.iterrows()
        }
        for pred in result.predictions:
            if not isinstance(pred.payload, ContinuousForecast):
                continue
            fd = pd.Timestamp(pred.forecast_date).normalize()
            actual = actual_by_date.get(fd)
            if actual is None:
                continue
            as_of = pd.Timestamp(pred.as_of)
            for h in task.horizons:
                if (as_of + offset * h).normalize() == fd:
                    horizon_errors[h].append(abs(pred.payload.point_forecast - actual))
                    break

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "80% CI Coverage": scores.get("coverage_80", float("nan")),
    }
    for h in eval_spec.tasks[0].horizons:
        errs = horizon_errors[h]
        row[f"MAE h={h}d"] = float(np.mean(errs)) if errs else float("nan")
    scorecard_rows.append(row)

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS")

print("━" * 72)
print(f"EVAL SCORECARD ({eval_spec.start} → {eval_spec.end}):")
print("━" * 72)
print(df_scorecard.to_string())

---
## 7. Diagnostics — reading past the leaderboard

The scorecard above is a single number per method. That hides *where* the score
comes from and *whether the ranking is even real*. The next cells decompose it
straight from the eval predictions — so they recompute on any rerun, smoke or
full:

- **CRPS by horizon** — does a method win everywhere, or is its mean dominated by
  one horizon? (For a short forecast, the 5-day calls are easy and nearly tied;
  the ranking is usually decided by the longest horizon.)
- **Mean CRPS ± standard error** — with only a handful of origins, are the gaps
  between methods bigger than the noise, or is the "winner" a coin flip?

With the **smoke spec (2 origins → a few scored points)** expect wide error bars
and an unstable ranking. That is exactly why a surprising leaderboard here is not
yet evidence of anything — it is a pipeline check.

In [ ]:
from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import (
    build_price_frame,
    eval_narrative_md,
    extract_agent_rationales,
    leaderboard_with_uncertainty,
    per_horizon_crps,
    predictions_to_frame,
)
from IPython.display import HTML, Markdown, display  # noqa: A004


# Explode every scored 2026 eval prediction into one tidy row per
# (predictor, origin, horizon): point, 80% interval, realised price, and CRPS.
# Everything in Sections 7–10 reads from this frame, so it all recomputes when
# you flip SMOKE_TEST off and rerun.
price_df = build_price_frame(data_service)
eval_frame = predictions_to_frame(eval_results, data_service)
eval_board = leaderboard_with_uncertainty(eval_frame)
ph_crps = per_horizon_crps(eval_frame)

print("━" * 72)
print("MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):")
print("━" * 72)
print(ph_crps.round(2).to_string())

In [ ]:
# Heatmap of the table above. Read it left-to-right: the short-horizon columns
# are usually a near-uniform green (everyone is right), and one long-horizon
# column carries the colour spread that sets the 'All' ranking.
viz.make_crps_heatmap(ph_crps)

In [ ]:
# Same leaderboard, now with a standard-error bar on each mean. If the bars of
# the top methods overlap, their ordering is not statistically distinguishable —
# the honest verdict when only a few origins have been scored.
viz.make_leaderboard_interval_chart(eval_board)

---
## 8. What are the top methods actually forecasting?

A CRPS number doesn't show *behaviour*. Below, each leading method's **median
forecast and 80% interval** are drawn against the realised WTI path at every
eval origin. This is where the leaderboard becomes legible — watch for who
tracks the move, who simply anchors to the last price, and whose intervals are
too narrow to cover the outcome when the market jumps.

In [ ]:
# Plot the leaderboard's top methods, and always include the best LLM/agent
# method for contrast (so the chart compares families even when a baseline leads).
_leaders = list(eval_board.index[:3])
_best_llm = next((p for p in eval_board.index if eval_board.loc[p, "family"] == "LLM / Agent"), None)
if _best_llm and _best_llm not in _leaders:
    _leaders.append(_best_llm)
print(f"Showing: {', '.join(_leaders)}")
viz.make_eval_forecast_chart(eval_frame, price_df, _leaders)

---
## 9. Reading the agent's reasoning

The news-reading agent attaches a free-text **rationale** to every forecast, and
a link to the full **Langfuse trace**. These are pulled straight from the
prediction metadata. This is where a surprising score becomes interpretable: you
can read whether the agent actually saw the geopolitical risk, and *how* it
turned that into a price and an interval — including, often, an interval far too
narrow for a regime shift.

In [ ]:
# One card per (agent, origin): the rationale, the per-horizon note, and a link
# to the full reasoning trace. Empty only if no LLM/agent predictor is enabled.
eval_rationales = extract_agent_rationales(eval_results)
display(HTML(viz.render_rationales_html(eval_rationales)))

---
## 10. Takeaways — computed from this run

The summary below is **generated from the eval results in memory, not
hard-coded**, so it always matches what actually ran: the real winner, whether
its lead clears the noise floor, the horizon that decided the ranking, the
best-performing family, and a calibration line. Flip `SMOKE_TEST` off, rerun,
and these takeaways update themselves with the full leaderboard.

In [ ]:
display(Markdown(eval_narrative_md(eval_frame, smoke=SMOKE_TEST)))

---
## 11. What stateless methods can't do

Sections 7–10 score and dissect this run on its own terms. But every method here
shares one structural limit, independent of who topped the leaderboard: it is
calibrated (or prompted) **once and never updated between rounds**. That is
intentional — it creates a clean baseline — but it leaves a systematic gap:

- **No error feedback.** If a method's intervals are consistently too narrow in
  an elevated-vol regime (read the coverage line in Section 10, and the squashed
  error bars in Section 8), it keeps making the same mistake. Nothing updates its
  calibration between origins.

- **No strategy evolution.** Each prediction starts from the same prior — the
  same fitted model, or the same prompt. Resolved outcomes disappear without
  influencing future forecasts.

- **Context without memory.** Even the news agent re-reads the world each origin;
  it does not accumulate what worked. The rationales in Section 9 are written
  fresh every time, with no record of how the last one resolved.

→ **Notebook 5** introduces adaptive agents that study the 2025 backtest, record
systematic observations, and calibrate their strategies accordingly. At inference
time, each agent receives the live stateless estimate and decides how to adjust
it — applying what it learned from training.

→ **Notebook 6** evaluates whether any training approach actually improved
out-of-sample performance on the held-out 2026 data — measured against the
stateless baseline this notebook just established.